# 2.5 — Basic RAG Pipeline

Now we connect everything:

```
Document → Chunk → Embed → Chroma
                                ↓
User Query → Retrieve chunks → Prompt → LLM → Answer
```

We'll build this end-to-end using LangChain LCEL (pipe operator `|`).

In [ ]:
!pip install langchain langchain-ollama langchain-community chromadb --quiet

## Step 1 — Load and Chunk the Document

In [ ]:
# Create the sample document
handbook = """
Acme Corp Employee Handbook

Section 1: Working Hours
Standard working hours are 9am to 5pm, Monday to Friday.
Employees may request flexible working hours with manager approval.
Overtime must be pre-approved and will be compensated at 1.5x the hourly rate.

Section 2: Leave Policy
All full-time employees receive 20 days of annual leave per year.
Sick leave is up to 10 days per year with a medical certificate.
Parental leave is 16 weeks fully paid for primary caregivers.
Leave requests must be submitted at least 2 weeks in advance.

Section 3: Remote Work
Employees may work remotely up to 3 days per week.
A reliable internet connection is required for remote work.
Remote workers must be available during core hours: 10am to 3pm.
All remote work equipment is provided by the company.

Section 4: Code of Conduct
Employees are expected to treat colleagues with respect and professionalism.
Harassment, discrimination, or bullying of any kind will not be tolerated.
Confidential company information must not be shared outside the organization.

Section 5: Benefits
Health insurance is provided for all full-time employees and their immediate family.
A gym membership subsidy of $50 per month is available.
Employees receive a $1,000 annual learning and development budget.
Free meals are provided in the office cafeteria.
"""

with open('handbook.txt', 'w') as f:
    f.write(handbook)

print('Document saved.')

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

loader = TextLoader('handbook.txt')
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(documents)

print(f'Loaded {len(documents)} document → split into {len(chunks)} chunks')

## Step 2 — Embed and Store

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model='llama3.2')

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory='./rag_db'
)

retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
print('Vector store ready.')

## Step 3 — Build the RAG Chain

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOllama(model='llama3.2', temperature=0)

prompt = ChatPromptTemplate.from_template("""
You are a helpful HR assistant. Answer the question using only the context below.
If the answer is not in the context, say "I don't have that information."

Context:
{context}

Question: {question}

Answer:
""")

def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

# RAG chain using LCEL
rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print('RAG chain ready.')

## Step 4 — Ask Questions

In [ ]:
questions = [
    'How many annual leave days do employees get?',
    'Can I work from home every day?',
    'What benefits are available?',
    'What happens if I work overtime?',
    'What is the company stock price?',  # not in the document
]

for question in questions:
    print(f'Q: {question}')
    answer = rag_chain.invoke(question)
    print(f'A: {answer}')
    print()

## Step 5 — Compare RAG vs No RAG

In [ ]:
question = 'How many days of annual leave do Acme Corp employees get?'

# Without RAG
plain_answer = llm.invoke(question).content

# With RAG
rag_answer = rag_chain.invoke(question)

print('WITHOUT RAG:')
print(plain_answer)
print()
print('WITH RAG:')
print(rag_answer)

## Summary

| Step | Code |
|------|------|
| Load | `TextLoader` / `PyPDFLoader` |
| Chunk | `RecursiveCharacterTextSplitter` |
| Embed + Store | `Chroma.from_documents()` |
| Retrieve | `vectorstore.as_retriever()` |
| Generate | `prompt | llm | StrOutputParser()` |
| Full chain | `{context: retriever, question: passthrough} | prompt | llm` |